In [ ]:
import torch
import csv 
import numpy as np

In [ ]:
wine_path = '../../data/p1ch4/tabular-wine/winequality-white.csv'
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=';',skiprows=1)
wineq_numpy

In [ ]:
col_list = next(csv.reader(open(wine_path),delimiter=';'))
wineq_numpy.shape, col_list

In [ ]:
wineq = torch.from_numpy(wineq_numpy)
wineq.shape, wineq.dtype

In [ ]:
# input data (without score)
data = wineq[:,:-1] # all rows and all but last column
data, data.shape

In [ ]:
target = wineq[:,-1]
target, target.shape

In [ ]:
# make integers as labels
target = wineq[:,-1].long()
target

In [ ]:
# One-hot encoding
# a bit better for categorical encoding
# for example, the scores MUST be integers, so we do not
# want the output of our models to yield 6.6, say.

target_onehot = torch.zeros(target.shape[0],10) # rows are examples, columns the 10 categories

# scatter_ is an inplace method
# the first argument is the dimension where the operation
# will be performed (here, that is along the columns)
# target.unsqueeze(1) makes the tensor effectively a column vector
# so each row has a number from this tensor, indicating the position
# for which to put in the value that is the final argument
target_onehot.scatter_(1, target.unsqueeze(1),1.0)

In [ ]:
''' 
Recall that unsqueeze looks at the shape array
and puts an extra singleton dimension in place
of where you have specified (here at index 1)
'''
target_unsqueeze = target.unsqueeze(1)
target_unsqueeze

In [ ]:
target.shape

In [ ]:
target_unsqueeze.shape

In [ ]:
data

In [ ]:
data_mean = torch.mean(data, dim=0) # dim=0 indicates we want the mean across the rows
data_mean

In [ ]:
data_var = torch.var(data, dim=0)
data_var

In [ ]:
data_normalized = (data - data_mean) / torch.sqrt(data_var)
data_normalized

In [ ]:
# Data Exploration
# First, let's look at bad wines

bad_indexes = target <= 3 # binary tensory
bad_indexes.shape, bad_indexes.dtype, bad_indexes.sum()

In [ ]:
bad_data = data[bad_indexes]
bad_data.shape

In [ ]:
bad_data = data[target <= 3]
mid_data = data[(target > 3) & (target < 7)]
good_data = data[target >= 7]

bad_mean = torch.mean(bad_data, dim=0)
mid_mean = torch.mean(mid_data, dim=0)
good_mean = torch.mean(good_data, dim=0)

for i, args in enumerate(zip(col_list, bad_mean, mid_mean, good_mean)):
    print('{:2} {:20} {:6.2f} {:6.2f} {:6.2f}'.format(i, *args))

In [ ]:
# crude model would be to use a threshold on total sulfur dioxide
total_sulfur_threshold = 141.83
total_sulfur_data = data[:,6]
# .lt is less than, same as total_sulfur_data < total_sulfur_threshold
predicted_indexes = torch.lt(total_sulfur_data, total_sulfur_threshold)

In [ ]:
predicted_indexes.shape, predicted_indexes.dtype, predicted_indexes.sum()

In [ ]:
# "actual" means "actually good"
actual_indexes = target > 5

actual_indexes.shape, actual_indexes.dtype, actual_indexes.sum()

Our simplistic threshold model provided 2727 wines that can be considered good while the true value is 3258. Therefore, we certainly have some false negatives. We have not even explored false positives at this point.

In [ ]:
# take the intersection of actually good wines and predicted wines via
# the boolean values for their indexes
n_matches = torch.sum(actual_indexes & predicted_indexes).item() 
n_predicted = torch.sum(predicted_indexes).item()
n_actual = torch.sum(actual_indexes).item()

# number of correct answers, percent of predictions that were correct, true positive rate
n_matches, n_matches/n_predicted, n_matches/n_actual